# Prompt 2: Execution Modes and Deploy Modes Deep Dive
**Databricks Certified Associate Developer for Apache Spark — Topic 1 (20%)**

---

## Overview

This notebook covers two related but distinct concepts that are tested on the exam:

| Concept | Question it answers |
|---|---|
| **Execution mode** | *Where does the Spark cluster run?* |
| **Deploy mode** | *Where does the Driver process run?* |

These are set independently. You can run on YARN (execution mode) with the driver on the
client machine (deploy mode = client), or with the driver inside the cluster (deploy mode = cluster).

---
## 1. Execution Modes — Where the Cluster Runs

### 1a. Local Mode
Everything runs in a **single JVM** on the submitting machine. No cluster required.

| Master string | Threads used | Use case |
|---|---|---|
| `local` | 1 thread | Unit tests, single-threaded debugging |
| `local[4]` | 4 threads | Development with limited parallelism |
| `local[*]` | All available cores | Local development, maximise CPU usage |

> **Exam tip**: In local mode there are **no Executors** — the Driver and "Executor" are the same JVM thread(s). There is no Cluster Manager involved.

### 1b. Standalone Mode
Spark's **built-in cluster manager**. No Hadoop required.
- Master URL: `spark://host:7077`
- Start with `./sbin/start-master.sh` and `./sbin/start-worker.sh`

### 1c. YARN Mode
Resource managed by **Hadoop YARN**.
- Master string: `yarn`
- Requires `HADOOP_CONF_DIR` or `YARN_CONF_DIR` set so Spark can find the YARN ResourceManager
- Best for organisations already running Hadoop

### 1d. Kubernetes Mode
Executors run as **pods** managed by Kubernetes.
- Master URL: `k8s://https://host:port`
- Each Executor is a separate container pod; Driver can also run as a pod (cluster mode)

### Execution Mode Comparison

| Mode | Cluster Manager | Best for | Requires |
|---|---|---|---|
| `local[*]` | None (single JVM) | Development / testing | Nothing |
| Standalone | Spark's own manager | Simple dedicated Spark clusters | Spark on all workers |
| YARN | Hadoop YARN | Hadoop ecosystems | Hadoop/YARN cluster |
| Kubernetes | Kubernetes | Cloud-native / containerised workloads | K8s cluster |

---
## 2. Deploy Modes — Where the Driver Runs

Deploy mode is **independent of execution mode** and is set with `--deploy-mode` on `spark-submit`.

```
┌──────────────────────────────────────────────────────────────┐
│                    CLIENT DEPLOY MODE                        │
│                                                              │
│  ┌──────────────────────┐        ┌──────────────────────┐   │
│  │   Client Machine     │        │      Cluster         │   │
│  │  ┌────────────────┐  │        │  ┌────────────────┐  │   │
│  │  │    DRIVER      │◄─┼────────┼─►│   Executor 1   │  │   │
│  │  │  (your JVM)    │  │        │  ├────────────────┤  │   │
│  │  └────────────────┘  │        │  │   Executor 2   │  │   │
│  │  stdout visible here │        │  └────────────────┘  │   │
│  └──────────────────────┘        └──────────────────────┘   │
│  ⚠ Client must stay alive for the duration of the job        │
└──────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────┐
│                   CLUSTER DEPLOY MODE                        │
│                                                              │
│  ┌──────────────┐         ┌──────────────────────────────┐  │
│  │    Client    │ submits │          Cluster             │  │
│  │   Machine    │────────►│  ┌──────────┐ ┌──────────┐   │  │
│  │              │         │  │  DRIVER  │ │Executor 1│   │  │
│  │  ✓ Can       │         │  │(worker   │ ├──────────┤   │  │
│  │  disconnect  │         │  │  node)   │ │Executor 2│   │  │
│  └──────────────┘         │  └──────────┘ └──────────┘   │  │
│                            └──────────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
```

| | Client mode | Cluster mode |
|---|---|---|
| Driver location | Submitting machine | Worker node inside cluster |
| stdout/stderr | Visible in terminal | Inside cluster logs |
| Client must stay alive? | **Yes** | **No** |
| Typical use | Interactive dev, debugging | Production scheduled jobs |
| Access Spark UI | Directly on client port | Via cluster proxy |

> **#1 Exam Question**: *"You submit a job with `--deploy-mode cluster`. Where does the driver run?"*  
> **Answer: On a worker node inside the cluster.**

---
## 3. spark-submit Command Structure

```bash
spark-submit \
  --master yarn \                    # execution mode
  --deploy-mode cluster \            # deploy mode
  --executor-memory 4g \             # memory per executor
  --executor-cores 2 \               # CPU cores per executor
  --num-executors 10 \               # number of executor JVMs
  --driver-memory 2g \               # memory for the driver
  --conf spark.sql.shuffle.partitions=200 \  # arbitrary config
  my_app.py                          # your application file
```

### Key spark-submit flags for the exam:

| Flag | Purpose |
|---|---|
| `--master` | Cluster manager URL or mode |
| `--deploy-mode` | `client` or `cluster` |
| `--executor-memory` | RAM per executor (e.g., `4g`) |
| `--executor-cores` | CPU cores per executor |
| `--num-executors` | Number of executor processes (YARN/Standalone) |
| `--driver-memory` | RAM for the driver process |
| `--conf key=value` | Any Spark config property |
| `--py-files` | Additional Python files to distribute |
| `--files` | Additional files to distribute to executors |
| `--packages` | Maven coordinates for dependencies |

In [ ]:
# Configuring execution mode in SparkSession.builder (Python code)
from pyspark.sql import SparkSession

# ── Local mode ──────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("ExecutionModeDemo")
    .master("local[*]")   # all cores on local machine
    .getOrCreate()
)
print(f"Master:              {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

In [ ]:
# Execution mode examples (commented — require real clusters)

# Standalone mode
# spark = (
#     SparkSession.builder
#     .appName("StandaloneDemo")
#     .master("spark://master-host:7077")
#     .config("spark.executor.memory", "4g")
#     .config("spark.executor.cores", "2")
#     .getOrCreate()
# )

# YARN mode
# spark = (
#     SparkSession.builder
#     .appName("YarnDemo")
#     .master("yarn")
#     .config("spark.executor.memory", "4g")
#     .config("spark.executor.instances", "10")
#     .getOrCreate()
# )

# Kubernetes mode
# spark = (
#     SparkSession.builder
#     .appName("K8sDemo")
#     .master("k8s://https://k8s-api-server:443")
#     .config("spark.kubernetes.container.image", "apache/spark:3.5.0")
#     .config("spark.executor.instances", "5")
#     .getOrCreate()
# )

# Best practice: set --master via spark-submit flag (not hardcoded)
# This keeps application code portable across environments
print("Best practice: set --master via spark-submit, not hardcoded in code")
print("This allows the same script to run locally and in production unchanged")

In [ ]:
# Inspecting the active SparkSession configuration
print("=== SparkContext info ===")
print(f"Master:               {spark.sparkContext.master}")
print(f"App name:             {spark.sparkContext.appName}")
print(f"Default parallelism:  {spark.sparkContext.defaultParallelism}")

print("\n=== Relevant config properties ===")
configs = [
    "spark.executor.memory",
    "spark.executor.cores",
    "spark.driver.memory",
    "spark.sql.shuffle.partitions",
]
for key in configs:
    try:
        print(f"  {key} = {spark.conf.get(key)}")
    except Exception:
        print(f"  {key} = (not set — using default)")

In [ ]:
# Demonstrating parallelism difference between local[1] and local[*]
import time

data = list(range(1_000_000))
rdd = spark.sparkContext.parallelize(data, numSlices=8)

start = time.time()
total = rdd.reduce(lambda a, b: a + b)
elapsed = time.time() - start

print(f"Sum:                   {total}")
print(f"Partitions (= Tasks):  {rdd.getNumPartitions()}")
print(f"Elapsed:               {elapsed:.3f}s")
print(f"Active cores (local[*]):{spark.sparkContext.defaultParallelism}")
print()
print("With local[*]: all cores run Tasks in parallel")
print("With local[1]: only 1 core, Tasks run sequentially")

---
## 4. Combining Execution Mode and Deploy Mode

These two settings are independent. Any combination is valid (where supported):

| Execution mode | Deploy mode | Typical scenario |
|---|---|---|
| `local[*]` | N/A (driver IS the cluster) | Development / unit testing |
| `standalone` | `client` | Ad-hoc queries from dev workstation |
| `standalone` | `cluster` | Scheduled jobs on dedicated Spark cluster |
| `yarn` | `client` | Interactive PySpark shell, Jupyter on edge node |
| `yarn` | `cluster` | Production ETL jobs submitted by scheduler |
| `k8s` | `cluster` | Cloud-native containerised pipelines |

> **Note**: `local` mode does not support `--deploy-mode cluster` — there is no cluster to deploy the driver into.

---
## 5. Real-World Scenarios

<details>
<summary>Scenario 1: Dev runs fine locally but fails on cluster — driver memory error</summary>

**Situation**: A data engineer develops a PySpark pipeline using `local[*]` on their laptop (16 GB RAM). After switching to `yarn` + `cluster` deploy mode in production, the job fails with `OutOfMemoryError` on the driver.

**Root cause**: In local mode the driver used the laptop's 16 GB. In cluster mode the default `spark.driver.memory` is 1 GB. The driver was calling `df.collect()` on a large result.

```bash
# Fix: explicitly set driver memory in spark-submit
spark-submit \
  --master yarn \
  --deploy-mode cluster \
  --driver-memory 8g \
  --executor-memory 4g \
  --executor-cores 2 \
  --num-executors 10 \
  my_pipeline.py
```

```python
# Better fix: avoid collect() on large DataFrames
# Instead of:
rows = df.collect()  # loads all data into driver RAM

# Use:
df.write.parquet("/output/result")  # stays distributed
df.show(20)  # sample only
```

**Expected outcome**: Setting `--driver-memory 8g` resolves OOM. Replacing `collect()` eliminates the root cause.

**Exam sub-topic**: Deploy mode; driver memory configuration
</details>

<details>
<summary>Scenario 2: Production job must survive client machine shutdown</summary>

**Situation**: A scheduled ETL job runs every night at 2 AM. Submitted with `--deploy-mode client` from a jump server. The job fails whenever the jump server reboots mid-run.

**Root cause**: In client mode, if the submitting machine shuts down, the Driver process dies and the job is killed.

```bash
# Problem: driver on this machine — job dies if machine goes down
spark-submit --master yarn --deploy-mode client my_etl_job.py

# Fix: cluster mode — driver lives inside the cluster
spark-submit --master yarn --deploy-mode cluster my_etl_job.py
# Client machine can now disconnect or reboot safely
```

**Expected outcome**: With cluster mode, the YARN ApplicationMaster (Spark Driver) runs on a worker node. Submitting machine can go offline without affecting the job.

**Exam sub-topic**: Client must stay alive vs cluster mode resilience
</details>

<details>
<summary>Scenario 3: Choosing local mode for CI/CD unit tests</summary>

**Situation**: A team wants to run automated PySpark unit tests in GitHub Actions without provisioning a real Spark cluster.

```python
# conftest.py — pytest fixture
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    return (
        SparkSession.builder
        .master("local[2]")                           # 2 threads, no cluster
        .appName("UnitTests")
        .config("spark.sql.shuffle.partitions", "2")  # small for tests
        .config("spark.ui.enabled", "false")          # no UI in CI
        .getOrCreate()
    )

# test_transformations.py
def test_filter_by_salary(spark):
    data = [("Alice", 90000), ("Bob", 50000)]
    df = spark.createDataFrame(data, ["name", "salary"])
    result = df.filter(df.salary > 60000)
    assert result.count() == 1
    assert result.first()["name"] == "Alice"
```

**Expected outcome**: Tests run in under 10 seconds on a CI runner with no cluster.

**Exam sub-topic**: Local execution mode; `local[n]` thread count
</details>

<details>
<summary>Scenario 4: Interactive PySpark on YARN edge node (client mode)</summary>

**Situation**: A data scientist wants to run interactive queries against a Hadoop cluster from a Jupyter notebook hosted on an edge node.

**Why client mode**: The notebook server is always running (satisfies "must stay alive"). Client mode gives immediate stdout feedback in the notebook.

```bash
# Start PySpark shell in YARN client mode
pyspark \
  --master yarn \
  --deploy-mode client \
  --executor-memory 8g \
  --executor-cores 4 \
  --num-executors 20
```

```python
# Or configure via environment in notebook startup
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--master yarn --deploy-mode client "
    "--executor-memory 8g --num-executors 10 pyspark-shell"
)
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("InteractiveSession").getOrCreate()
# Driver runs on the edge node; Executors run inside YARN cluster
```

**Expected outcome**: Interactive session with results directly in the notebook. Spark UI accessible on localhost on the edge node.

**Exam sub-topic**: Client deploy mode; stdout visibility; interactive development
</details>

<details>
<summary>Scenario 5: Kubernetes cluster mode — driver as a pod</summary>

**Situation**: A cloud team runs all workloads on Kubernetes and needs to submit a Spark job where both driver and executors run as K8s pods.

```bash
spark-submit \
  --master k8s://https://k8s-api:443 \
  --deploy-mode cluster \
  --name my-spark-job \
  --conf spark.kubernetes.container.image=apache/spark:3.5.0 \
  --conf spark.kubernetes.namespace=spark-jobs \
  --conf spark.executor.instances=5 \
  --conf spark.executor.memory=4g \
  --conf spark.executor.cores=2 \
  local:///opt/spark/examples/jars/spark-examples.jar
```

```bash
# Monitor pods
kubectl get pods -n spark-jobs
# my-spark-job-driver    1/1  Running
# my-spark-job-exec-1    1/1  Running
# my-spark-job-exec-2    1/1  Running

# Stream driver logs
kubectl logs -f my-spark-job-driver -n spark-jobs
```

**Expected outcome**: Driver pod appears in K8s namespace; executor pods are dynamically created and destroyed. Submitting machine can disconnect after submission.

**Exam sub-topic**: Kubernetes execution mode; cluster deploy mode; client disconnection
</details>

---
## 6. Quick Reference Summary

| Setting | Flag | Options | What it controls |
|---|---|---|---|
| Execution mode | `--master` | `local[*]`, `spark://`, `yarn`, `k8s://` | Where the Spark cluster runs |
| Deploy mode | `--deploy-mode` | `client`, `cluster` | Where the Driver JVM runs |

### Exam Cheat Sheet

- `local[*]` → single JVM, all cores, no cluster manager
- `--deploy-mode client` → driver on submitting machine, stdout visible, machine **must stay up**
- `--deploy-mode cluster` → driver on worker node, **client can disconnect**, used for production
- YARN client mode = interactive/debug; YARN cluster mode = production jobs
- Kubernetes only supports `cluster` deploy mode

---
## 7. Exam Practice Questions

**Q1**: A Spark job is submitted with `--master yarn --deploy-mode cluster`. Where does the Driver run?

<details><summary>Answer</summary>
On a **worker node inside the YARN cluster** (as the YARN ApplicationMaster). The submitting machine does not need to remain connected.
</details>

---

**Q2**: Which master string uses all available CPU cores on the local machine?

<details><summary>Answer</summary>
`local[*]` — uses all available CPU cores as separate threads in a single JVM.
</details>

---

**Q3**: A developer submits a job with `--deploy-mode client` and their laptop goes to sleep mid-job. What happens?

<details><summary>Answer</summary>
The **job fails**. In client mode the Driver runs on the client machine. When the machine sleeps, the Driver JVM pauses/dies and the Executors lose contact with the Driver, killing the job.
</details>

---

**Q4**: What is the difference between `local[1]` and `local[4]`?

<details><summary>Answer</summary>
`local[1]` uses **1 thread** (sequential, no parallelism). `local[4]` uses **4 threads** (up to 4 tasks run concurrently). Both run in a single JVM with no cluster manager.
</details>

---

**Q5**: Which deploy mode would you choose for a production ETL job scheduled by Apache Airflow on YARN? Why?

<details><summary>Answer</summary>
**Cluster mode** (`--deploy-mode cluster`). The Airflow worker submitting the job does not need to stay alive for the job's duration. The Driver runs inside the cluster, making the job resilient to Airflow worker restarts.
</details>